# Seminar 1. Python Survival Kit

A quick Python refresher and language features that will help you implement algorithms.

**By the end of this session, you will be able to:** transform collections, iterate over related data, explain common reference-related bugs, customize sorting, and distinguish an eager list from a lazy computation.

Python 3 is required; no third-party libraries are needed. Run the cells from top to bottom (`Shift+Enter`). When a cell is preceded by a question, predict the result before running it. To repeat the session from a clean state, restart the kernel and run all cells.

Roadmap: program execution → syntax refresher → comprehensions → `enumerate` / `zip` → unpacking → references and mutability → functions and sorting → generators → cheat sheet.


## 1. What happens when we call a function?

**Discuss:** what does the computer do when we call `f(10)`? What result do we expect, and is “multiply by two and add one” a complete explanation?


In [ ]:
def f(x):
    return x * 2 + 1

print(f(10))


Let's inspect the bytecode instructions using the standard `dis` module. You do not need to memorize their names yet: the goal is to see an intermediate layer between source code and execution.


In [ ]:
import dis
import platform
import sys

print('Implementation:', platform.python_implementation())
print('Version:', sys.version.split()[0])
dis.dis(f)


For typical execution in **CPython**, this simplified diagram is useful:

```text
Python source code
        ↓ parsing and compilation
Bytecode
        ↓ executed by the CPython interpreter
Machine instructions of the interpreter and called libraries
        ↓
CPU
```

The operating system manages the process, memory, and access to devices. It is not a separate mandatory stage that translates each bytecode instruction into machine code.

Bytecode and `dis` output may differ across CPython versions; other Python implementations may execute programs differently. We will revisit these layers throughout the course.


## 2. A quick syntax refresher

Variables bind names to values; conditions select actions, loops repeat them, and functions let us name a computation. This section is a brief check of familiar constructs.


In [ ]:
x = 42
if x > 10:
    print('Greater than ten:', x)

values = [1, 2, 3]
for value in values:
    print(value)

def increment(value):
    return value + 1

print('Function result:', increment(10))


In [ ]:
# Else in `for`:
x = [1, 2, 3, 4, 5]

for v in x:
    if v < 0:
        print('NEGATIVE', v)
        break
else:
    print('No negatives')

In [ ]:
numbers = [1, 2, 3]                  # list: ordered, allows duplicates
point = (3, 4)                       # tuple: immutable sequence
scores = {'Alice': 10, 'Bob': 8}     # dictionary: key → value
unique_numbers = {1, 2, 2, 3}        # set: unique elements

print(numbers[0])
print(point)

key = 'Bill'
v = scores.get(key, 0)
print(v)
print(unique_numbers)        # sort for predictable output


**Check your understanding:** what is the first index in a list? What happens if you access a missing dictionary key using square brackets? How many elements are in `unique_numbers`?

If basic syntax is still unfamiliar, start your independent review here: change the data and explain each line of output.


## 3. List comprehensions: transforming and filtering

First, let's collect squares using a regular loop.


In [ ]:
squares = []
for x in range(100):
    squares.append(x * x)

print(squares)


We can express the same result using a single **list comprehension**:

```python
[expression for item in iterable if condition]
```

The condition is optional. On each iteration, the condition is checked first, then the value to include is computed.


In [ ]:
squares = [x * x for x in range(10)]
print(squares)

even_squares = [
    x * x
    for x in range(20)
    if x % 2 == 0
]
print(even_squares)


**Before running the next cell:** how many keys will the dictionary contain, and how many elements will the set contain? Why does a repeated word not create a second key?


In [ ]:
words = ['pear', 'watermelon', 'apple', 'pear']

l = {}
for w in words:
    l[w] = len(w)

lengths = {word: len(word) for word in words}
first_letters = {word[0] for word in words}

print(lengths)
print(sorted(first_letters))


A dictionary comprehension builds `key: value` pairs; a set comprehension collects unique values. All words here are nonempty: an empty string has no element at index `0`.

Comprehensions work well for short transformations and filters. If the logic involves many steps and branches, a regular loop may be easier to read.


## 4. `enumerate` and `zip`

**Quick warm-up:** how can you rewrite the following loop without accessing the list by index?


In [ ]:
names = ['Alice', 'Bob', 'Charlie']
for i in range(len(names)):
    print(i, names[i])


In [ ]:
for i, name in enumerate(names):
    print(i, name)

`enumerate` yields `(index, item)` pairs. You can specify the starting number with `start`.


In [ ]:
names = ['Alice', 'Bob', 'Charlie']
for i, name in enumerate(names):
    print(i, name)

print('Numbering from one:')
for i, name in enumerate(names, start=1):
    print(i, name)


`zip` lets us iterate over several sources together. Each iteration produces a tuple of corresponding elements.


In [ ]:
names = ['Alice', 'Bob', 'Bill']
scores = [84, 97, 100]
email = ['a', 'b', 'c']

for name, score, mail in zip(names, scores, email):
    print(name, score, mail)

**Predict:** how many pairs will we get if the lists have different lengths?


In [ ]:
print(list(zip(['Alice', 'Bob', 'Charlie'], [84, 97])))


By default, `zip` stops when the shortest source is exhausted. If every item must have a match, you need to account for equal lengths separately.


## 5. Unpacking

Unpacking binds several names to the elements of a sequence. We have already used it in loops with `enumerate` and `zip`.


In [ ]:
a, b = 10, 20
a, b = b, a
print('After swapping:', a, b)

point = (3, 4)
x, y = point
print('Coordinates:', x, y)

values = [10, 20, 30, 40, 50]
first, *middle, last = values
print('First:', first, 'middle:', middle, 'last:', last)


In [ ]:
def f(x):
    return x, x**2, x**3, x**4

x1, x2, x3, x4 = f(2)

In [ ]:
values = [10, 20, 30, 40, 50]
first, first2, *middle, last2, last = values
print(first)
print(first2)
print(middle)
print(last2)
print(last)

The right-hand side of an assignment is evaluated before the names on the left are bound, so swapping does not require a temporary variable. A starred target collects the remaining elements into a list.

**Discuss:** what will `middle` contain if `values` has only two elements? Why will this unpacking fail with just one element?


In [ ]:
edges = [('B', 5), ('C', 2), ('D', 8)]
for vertex, weight in edges:
    print('Vertex:', vertex, 'weight:', weight)


## 6. Variables are names bound to objects

**Before running:** what will `a` contain after we append an element through `b`?


In [ ]:
a = [1, 2, 3]
b = a.copy()
b.append(4)

print('a:', a) # [1, 2, 3]
print('b:', b) # [1, 2, 3, 4]
print('Same object?', a is b)


The assignment `b = a` does not copy the list. Two names refer to the same object:

```text
a ─────┐
       ▼
    [1, 2, 3, 4]
       ▲
b ─────┘
```

`==` compares values; `is` checks whether two references point to the same object. Use `==` to compare numbers and strings by value.


In [ ]:
a = [1, 2, 3]
b = a.copy()

print('Before the change: equal?', a == b, 'same object?', a is b)
b.append(4)
print('a:', a)
print('b:', b)


`copy()` creates a new outer list. This is a **shallow copy**: the objects referenced by its elements are not themselves copied. This distinction matters for nested lists.

### Creating List with N values (1D array)

In [ ]:
N = 100
array = [0] * N
print(array[80])

### Creating 2D Array (matrix)

**Predict the result:** how many ones will appear in the matrix?

In [ ]:
matrix = [[0] * 3] * 3
matrix[0][0] = 1

for row in matrix:
    print(row)
print('Are the first two rows the same object?', matrix[0] is matrix[1])


The outer multiplication repeated a reference to the same row list three times. The change is visible through each of those references.

Let's create a separate list on each iteration:


In [ ]:
matrix = [[0] * 3 for _ in range(3)]
matrix[0][0] = 1

for row in matrix:
    print(row)
print('Are the first two rows the same object?', matrix[0] is matrix[1])

# Let's make it NxN:
N = 10
x = [[0] * N for i in range(N)]
x[0][0] = 1
for line in x:
    print(line)

Here, `_` is an ordinary variable name conventionally used for a value we do not need.

**Discuss:** why is `[0] * 3` fine inside a row? Numbers are immutable: `matrix[0][0] = 1` replaces a reference in the list rather than changing the object `0` itself.

### A mutable default argument

**Predict the three lines of output.** The function definition and calls are in the same cell so that rerunning it starts the experiment afresh.


In [ ]:
def add(x, result=[]):
    result.append(x)
    return result

print(add(1)) # [1]
print(add(2)) # [2]
print(add(3)) # [3]
x = []
print(add(4, x)) # [4]


A default argument value is evaluated when the function definition is executed. Calls that omit the second argument therefore share one list.

If each such call needs a fresh list, we can create it inside the function. We use `None` to indicate that no list was supplied.


In [ ]:
def add_fresh(x, result=None):
    if result is None:
        result = []
    result.append(x)
    return result

print(add_fresh(1))
print(add_fresh(2))
print(add_fresh(3))

saved = []
add_fresh(10, saved)
print('Explicitly supplied list:', saved)


This version still modifies an explicitly supplied list—that is the behavior chosen for this function. The `is None` check distinguishes an omitted list from an explicitly supplied empty list.

References and mutability will be useful when we discuss arguments, classes, copying, and memory management.


## 7. Functions as values and sorting with `key`

A function can be bound to another name and passed to another function. Parentheses call the function; without parentheses, we refer to the function object itself.


In [ ]:
def square(x):
    return x * x

operation = square
print(operation(10))
print(operation is square)


`sorted` returns a new list. Its `key` parameter accepts a function that computes a comparison value for each element.

**Predict the order:** how will the words be sorted by length?


In [ ]:
words = ['pear', 'watermelon', 'apple']
print(sorted(words, key=len))
print('Original list:', words)

def letter2(w):
    return w[1]
print('Now we can sort the words according to the 2nd letter:')
print(sorted(words, key=letter2))
# Or the same:
print(sorted(words, key=lambda w: w[1]))


In [ ]:
students = [('Alice', 84), ('Bob', 97), ('Charlie', 76)]

print('By increasing score:')
print(sorted(students, key=lambda student: student[1]))
print('By decreasing score:')
print(sorted(students, key=lambda student: student[1], reverse=True))


`lambda student: student[1]` is a short function that returns the score. We can replace it with a regular function:


In [ ]:
def get_score(student):
    return student[1]

print(sorted(students, key=get_score))


**Discuss:** why do we pass `key=get_score` rather than `key=get_score()`? How would you change the key function to sort by name?


## 8. (Additional) Generators: computing values on demand

A list comprehension immediately builds a list of all results. A generator expression in parentheses produces values as they are requested.

**Before running:** what has already been computed after each object is created?


In [ ]:
import sys

squares_list = [x * x for x in range(1000000)]
squares_gen = (x * x for x in range(1000000))

print('List object size:', sys.getsizeof(squares_list), 'bytes')
print('Generator object size:', sys.getsizeof(squares_gen), 'bytes')


`sys.getsizeof` reports the size of the object itself, not the total memory occupied by all objects it references. In particular, the list size shown here does not include the sizes of all its numbers. Exact values depend on the implementation and environment.

This example illustrates the difference between a materialized container and an object that stores computation state. It does not measure total program memory or prove that generators are always faster.


In [ ]:
# A fresh generator for a separate experiment.
squares_gen = (x * x for x in range(5))
print(next(squares_gen))
print(next(squares_gen))
print('Remaining values:', list(squares_gen))
print('After exhaustion:', list(squares_gen))

# The large list from the previous example is no longer needed.
del squares_list


A generator resumes where it left off and does not restart once exhausted. To make another pass, create a new generator.

**Discuss:** what changes if we immediately call `list` on the generator of a million squares? At what point will all the squares be computed?

For now, focus on the idea of lazy evaluation. Writing your own generator with `yield` is an optional exercise in the practice section.


## 9. A cheat sheet (may be useful for algorithms)

| Construct | Purpose |
| --- | --- |
| `1 in a` | Check does `a` contain value `1` (a may be list, tuple, set, dict...) |
| `enumerate(a)` | Index and element |
| `zip(a, b)` | Corresponding elements from two sources |
| `min(a)`, `max(a)`, `sum(a)` | Minimum, maximum, sum |
| `sorted(a, key=...)` | A new list in a chosen order |
| `any(...)`, `all(...)` | At least one / all conditions are true |
| `a[:i]` | First i values of a |
| `a[-i:]` | Last i values of a |
| `a[i:j]` | Values with indices from i to j-1 |
| `a[::-1]` | values of a in reverse order |
| `set(a)` | Unique elements |
| `d.get(key, default)` | The value for a key, or a fallback value |

You can modify the examples below during independent review.


In [ ]:
numbers = [3, -1, 3, 2]
print('Minimum, maximum, sum:', min(numbers), max(numbers), sum(numbers))
print('Sorted:', sorted(numbers))
print('Reverse order:', numbers[::-1])
print('Unique values, sorted for display:', sorted(set(numbers)))
print('Any negative values?', any(x < 0 for x in numbers))
print('All positive?', all(x > 0 for x in numbers))

scores = {'Alice': 10}
print('Alice:', scores.get('Alice', 0))
print('Bob:', scores.get('Bob', 0))
print('Dictionary after get:', scores)


`get` does not add a missing key to the dictionary. For an empty iterable, `sum` returns `0`, `any` returns `False`, and `all` returns `True`. Calling `min([])` or `max([])` without a `default` raises `ValueError`.

**Discussion question:** why is “all elements satisfy the condition” considered true for an empty sequence?


### P.S. Fun fact to discuss during the next webinar:

In [ ]:
f1 = lambda x: x * x
x1 = 10
x2 = x1
print(f1(x1) is f1(x2))

In [ ]:
f1 = lambda x: x * x
x1 = 100
x2 = x1
print(f1(x1) is f1(x2))


WOW!

WHAT?

WHY?